In [ ]:
import os
import time
import math
import json
import random
import pandas as pd
import numpy as np
import scipy as sp
import pylab as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

In [ ]:
RESULTS_PATH = "../data/results/baseline_2025-06-26/research-article_aimrd_f_run2"
secs = next(os.walk(RESULTS_PATH))[1]

#### frequency projection for all years

In [ ]:
yearly_freqs, yearly_diffs, yearly_ratios = utils.get_frequency_projection_yearly(
    RESULTS_PATH, "abstract"
)

In [ ]:
target_years = range(2013, 2026)
excess_words = {}
excess_counts = np.zeros_like(target_years)

for i, year in enumerate(target_years):
    freqs = yearly_freqs[str(year)]
    diffs = yearly_diffs[str(year)]
    ratios = yearly_ratios[str(year)]

    ind = np.log10(ratios) > np.log10(2) - (np.log10(freqs) + 4) * (np.log10(2) / 4)
    ind |= diffs > 0.01
    ind &= np.asarray(list(map(lambda x: x not in utils.ignore_list, freqs.index)))
    ind &= freqs >= 1e-3

    excess_words[str(year)] = ratios.index[ind]
    excess_counts[i] = np.sum(ind)
    print(f"{year}: found {np.sum(ind)} excess words. ")

In [ ]:
fig = plt.figure(figsize=(1.75, 2), layout="constrained")
plt.bar(target_years, excess_counts, color="#aaaaaa")
plt.xticks([2014, 2016, 2018, 2020, 2022, 2024], rotation=60)
plt.ylim([0, 500])

plt.savefig(os.path.join("../results/plots", "excess_words_abstract.png"), dpi=300)

In [ ]:
all_excess_words = []
for exc in excess_words.values():
    all_excess_words += list(exc)

all_excess_words = np.unique(all_excess_words)

print(f"Found {all_excess_words.size} unique excess words overall")

In [ ]:
excess_words["2018"][:100]

In [ ]:
all_excess_words

In [ ]:
def yearplot(year, freqs, diffs, ratios, cutoff):

    allowed_words = np.asarray(
        list(map(lambda x: x not in utils.ignore_list, freqs[str(year)].index))
    )
    allowed_words &= freqs[str(year)] >= cutoff

    freqs = freqs[str(year)][allowed_words]
    diffs = diffs[str(year)][allowed_words]
    ratios = ratios[str(year)][allowed_words]
    words = freqs.index

    ratios[ratios > 90] = 90
    diffs[diffs > 0.049] = 0.049

    fig = plt.figure(figsize=(7.2, 2.5), layout="constrained")
    plt.subplot(121)
    plt.scatter(freqs[ratios > 1], ratios[ratios > 1], s=2, rasterized=True)
    plt.ylim([1, 100])
    plt.xlim([1e-4, 1])
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(f"Frequency in {year}")
    plt.ylabel(f"Frequency ratio between {year} and {year - 2}")
    plt.plot([1e-4, 1], [2, 1], "k--", linewidth=0.5)

    txts_a = []
    txts_points_a = []
    show_labels = np.log10(ratios) > np.log10(2.2) - (np.log10(freqs) + 4) * (
        np.log10(2) / 4
    )
    ind_show_labels = np.where(show_labels)[0]

    for i in ind_show_labels:
        txt = plt.text(freqs[i], ratios[i], " " + words[i], fontsize=6, va="center")
        txts_a.append(txt)
        txt = plt.text(freqs[i], ratios[i], " ", fontsize=6, va="center")
        txts_points_a.append(txt)

    plt.gca().spines[["right", "top"]].set_visible(False)

    plt.subplot(122)
    plt.scatter(freqs[diffs > 0], diffs[diffs > 0], s=2, rasterized=True)
    plt.gca().spines[["right", "top"]].set_visible(False)
    plt.xlim([1e-4, 1])
    plt.xscale("log")
    plt.ylim([0, 0.05])
    plt.plot([1e-4, 1], [0.01, 0.01], "k--", linewidth=0.5)
    plt.xlabel(f"Frequency in {year}")
    plt.ylabel(f"Frequency gap between {year} and {year - 2}")

    txts_b = []
    txts_points_b = []
    for i in np.where(diffs > 0.011)[0]:
        txt = plt.text(
            freqs[i], diffs[i], words[i] + " ", fontsize=6, ha="right", va="center"
        )
        txts_b.append(txt)
        txt = plt.text(freqs[i], diffs[i], " ", fontsize=6, ha="right", va="center")
        txts_points_b.append(txt)

    def cleanup_labels(txts, txts_points, ha="left"):
        removed = np.zeros(len(txts), dtype=bool)
        bbox = txts_points[0].get_window_extent()
        space_width = bbox.x1 - bbox.x0
        for i in range(len(txts)):
            bbox = txts[i].get_window_extent()
            if ha == "left":
                bbox.x0 += space_width
            else:
                bbox.x1 -= space_width
            for j in range(len(txts)):
                if j != i:
                    bbox_point = txts_points[j].get_window_extent()
                    bbox_point.y0 += (bbox_point.y1 - bbox_point.y0) / 3
                    bbox_point.y1 -= (bbox_point.y1 - bbox_point.y0) / 3
                    if bbox.overlaps(bbox_point):
                        removed[i] = True
                        txts[i].remove()
                        break

    if len(txts_a) > 0:
        cleanup_labels(txts_a, txts_points_a)
    if len(txts_b) > 0:
        cleanup_labels(txts_b, txts_points_b, ha="right")

    fig.canvas.draw()

In [ ]:
for year in range(2013, 2026):
    yearplot(
        year, freqs=yearly_freqs, diffs=yearly_diffs, ratios=yearly_ratios, cutoff=1e-3
    )